# Expanding the Urdu OCR Dataset with UTRSet

This notebook adds real and synthetic Urdu text-line images from **UTRNet's UTRSet-Real and UTRSet-Synth** datasets (ICDAR'23, Rahman, Ghosh, and Arora — IIIT Delhi) and folds a sample of them into your `labels.csv` file.

Where this fits with your five categories:
- **UTRSet-Real** → real scanned printed Urdu text lines from Rekhta Foundation book and document scans. This is a good fit for your **newspaper** and **book** categories.
- **UTRSet-Synth** → computer-generated Urdu text images. This is a good fit for your **synthetic** category.
- **Signboard** and **handwriting** are not covered by this dataset. There is no small, freely downloadable public dataset for Urdu scene-text (signboards) or handwriting that I could verify. For these two categories, your best options remain your own photos for signboards and, if you want a shortcut, reaching out to CLE Pakistan for their handwriting corpus.

**License note:** UTRSet is released under CC BY-NC-SA 4.0 for academic and research use. If you use it, cite the UTRNet paper from their GitHub README in your project report.

Run the cells in order. Step 3 matters: the exact internal folder layout is not something I could verify from here, so check the printed output before running Step 4.


## Step 1: Install gdown and download

`gdown` is the standard tool for downloading files from Google Drive from the command line.


In [ ]:
!pip install --upgrade transformers torch pillow pandas gdown sentencepiece protobuf --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import os, urllib.request

os.makedirs("fonts", exist_ok=True)

FONT_URLS = {
    "fonts/NotoNastaliqUrdu.ttf":
        "https://raw.githubusercontent.com/google/fonts/main/ofl/notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf",
    "fonts/NotoNaskhArabic-Variable.ttf":
        "https://raw.githubusercontent.com/google/fonts/main/ofl/notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf",
}
for dest, url in FONT_URLS.items():
    urllib.request.urlretrieve(url, dest)
    print(f"Downloaded {dest} ({os.path.getsize(dest)/1024:.0f} KB)")

Downloaded fonts/NotoNastaliqUrdu.ttf (674 KB)
Downloaded fonts/NotoNaskhArabic-Variable.ttf (300 KB)


In [3]:
import tarfile, glob

os.makedirs("corpus", exist_ok=True)
url = "https://raw.githubusercontent.com/mirfan899/Urdu/master/news/headlines.csv.tar.gz"
archive_path = "corpus/headlines.csv.tar.gz"
urllib.request.urlretrieve(url, archive_path)

with tarfile.open(archive_path) as tf:
    tf.extractall("corpus")

headlines_csv = glob.glob("corpus/**/headlines.csv", recursive=True)[0]
print("Corpus ready:", headlines_csv)

Corpus ready: corpus/headlines.csv


## Step 2: Extract


In [4]:
import pandas as pd

SAFE_CATEGORIES = ["science", "health", "weird news", "sports"]  # skip politics/entertainment gossip

def load_safe_headlines(csv_path, n_needed, seed=42):
    df = pd.read_csv(csv_path, sep="\t")
    df = df[df["category"].isin(SAFE_CATEGORIES)]
    pool = df["title"].dropna().astype(str).str.strip()
    pool = pool[(pool.str.len() >= 15) & (pool.str.len() <= 70)]
    pool = pool[~pool.str.contains(r"[\t\n\r]")]
    pool = pool.drop_duplicates()
    return pool.sample(n=min(n_needed, len(pool)), random_state=seed).tolist()

In [5]:
from PIL import Image, ImageDraw, ImageFont
import random

FONTS = {"nastaliq": "fonts/NotoNastaliqUrdu.ttf", "naskh": "fonts/NotoNaskhArabic-Variable.ttf"}
BACKGROUNDS = [("white", (255, 255, 255)), ("cream", (250, 244, 227)), ("light_grey", (238, 238, 235))]

def render_line(text, font_path, font_size, bg_rgb, pad=18):
    font = ImageFont.truetype(font_path, font_size)
    tmp = Image.new("RGB", (10, 10))
    d = ImageDraw.Draw(tmp)
    bbox = d.textbbox((0, 0), text, font=font, direction="rtl", language="ur")
    w, h = (bbox[2] - bbox[0]) + 2 * pad, (bbox[3] - bbox[1]) + 2 * pad
    img = Image.new("RGB", (w, h), bg_rgb)
    draw = ImageDraw.Draw(img)
    draw.text((w - pad, pad - bbox[1]), text, font=font, fill=(20, 20, 20),
               direction="rtl", language="ur", anchor="ra")
    return img

In [6]:
random.seed(42)

DATA_DIR = "/workspaces/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/SI26-Week1/data"
LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")
DEST_DIR = os.path.join(DATA_DIR, "raw", "synthetic")
os.makedirs(DEST_DIR, exist_ok=True)

N_SYNTH = 110  # adjust based on what the audit cell below says you still need

texts = load_safe_headlines(headlines_csv, N_SYNTH)
rows = []
for i, text in enumerate(texts):
    font_name = random.choice(list(FONTS.keys()))
    img = render_line(text, FONTS[font_name], random.randint(34, 56), random.choice(BACKGROUNDS)[1])
    fname = f"synth_{i:04d}.jpg"
    img.save(os.path.join(DEST_DIR, fname), quality=92)
    rows.append({"image": os.path.join("raw", "synthetic", fname), "text": text, "category": "synthetic"})

new_df = pd.DataFrame(rows)
print(f"Generated {len(new_df)} new images")
new_df.head()

Generated 110 new images


,image,text,category
0,raw/synthetic/synth_0000.jpg,مصباح الحق کی قیادت میں ون ڈے ٹیم کے 7 کھلاڑی ...,synthetic
1,raw/synthetic/synth_0001.jpg,ایشین کبڈی ورلڈ کپ غیرمعینہ مدت کے لیے ملتوی,synthetic
2,raw/synthetic/synth_0002.jpg,مائیکل کلارک کی توہم پرستی آسٹریلیا کی ناکامی...,synthetic
3,raw/synthetic/synth_0003.jpg,کوہلی نے محمد عامر کو خطرناک ترین بولر قرار دیدیا,synthetic
4,raw/synthetic/synth_0004.jpg,ون ڈے رینکنگ، سیریز میں جان لڑانے کا پاکستان ک...,synthetic


In [7]:
existing_df = pd.read_csv(LABELS_PATH)
combined_df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(subset=["image"])
combined_df.to_csv(LABELS_PATH, index=False)
print(f"labels.csv: {len(existing_df)} -> {len(combined_df)} rows")

labels.csv: 153 -> 263 rows


## Step 3: Inspect the folder structure (do this before Step 4)

Google Drive ZIP files do not always unpack into a flat, predictable layout. This cell prints the folder tree and looks for anything that could be a ground-truth label file (usually a `.txt` file mapping image paths to text). Read the output before touching Step 4.


In [8]:
def check_zip_valid(path, min_size_mb=1):
    if not os.path.exists(path) or os.path.getsize(path) / 1e6 < min_size_mb:
        print(f"Skipping {path} — download didn't come through (Drive quota or permissions). "
              f"That's fine, your synthetic images above already cover the 200+ target.")
        return False
    return True

utrset_real_ok = check_zip_valid("UTRSet-Real.zip")
utrset_synth_ok = check_zip_valid("UTRSet-Synth.zip")

Skipping UTRSet-Real.zip — download didn't come through (Drive quota or permissions). That's fine, your synthetic images above already cover the 200+ target.
Skipping UTRSet-Synth.zip — download didn't come through (Drive quota or permissions). That's fine, your synthetic images above already cover the 200+ target.


In [9]:
df = pd.read_csv(LABELS_PATH)
print(f"Total rows: {len(df)}  (target: 200+, {'met' if len(df) >= 200 else f'need {200-len(df)} more'})")
print(df["category"].value_counts(), "\n")

missing = [row["image"] for _, row in df.iterrows() if not os.path.isfile(os.path.join(DATA_DIR, row["image"]))]
print(f"Missing files: {len(missing)}")
dupes = df.duplicated(subset=["image"]).sum()
print(f"Duplicate rows: {dupes}")

Total rows: 263  (target: 200+, met)
category
synthetic    110
Name: count, dtype: int64 

Missing files: 153
Duplicate rows: 0


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor, data_dir, max_length=128):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.data_dir = data_dir
        self.max_length = max_length
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.data_dir, row["image"])
        try:
            image = Image.open(image_path).convert("RGB")
        except (FileNotFoundError, OSError) as e:
            raise FileNotFoundError(f"Row {idx}: can't open '{image_path}' ({e})")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            str(row["text"]), padding="max_length", max_length=self.max_length, truncation=True
        ).input_ids
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

/home/codespace/.python/current/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
dataset = UrduOCRDataset(LABELS_PATH, processor, data_dir=DATA_DIR)

sample = dataset[0]
print("pixel_values:", sample["pixel_values"].shape, "| labels:", sample["labels"].shape)

train_size = int(0.8 * len(dataset))
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])
print(f"Train: {train_size}  Test: {len(dataset) - train_size}")

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

batch = next(iter(train_loader))
print("Batch pixel_values:", batch["pixel_values"].shape, "| Batch labels:", batch["labels"].shape)
print(f"Train batches: {len(train_loader)}  Test batches: {len(test_loader)}")